<a href="https://colab.research.google.com/github/paulchilemya2024-beep/DATA-SCIENCE-CS250-/blob/main/Ch8_Evaluating_Model_Performance_Activity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 8 In-Class Activity: Evaluating Model Performance

**zyBooks: Data Science Foundations with Python, Ch. 8 (sections 8.1–8.6, 8.8)** · About 30 minutes · Work in pairs

**Names:** Munyaradzi Maponga and Paul Chilemya

In this activity you will split data, measure regression and classification error, and use cross-validation to pick the better model.

| Part | Topic | Time |
| --- | --- | --- |
| 1 | Train, validation, and test sets; overfitting | 7 min |
| 2 | Regression loss: MAE, MSE, RMSE | 6 min |
| 3 | Binary classification metrics | 9 min |
| 4 | Cross-validation and comparing models | 6 min |
| — | Exit questions | 2 min |

**Before you start:** click **File → Save a copy in Drive** so your answers are saved.

**To submit:** run every code cell (**Runtime → Run all**), answer every question in its answer cell, then **File → Download → Download .ipynb** and upload it.

## Setup
Run this cell first. Both datasets ship with scikit-learn, so nothing needs to be downloaded.

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
    confusion_matrix, accuracy_score, precision_score, recall_score, f1_score)
print("Setup complete")

Setup complete


## Part 1: Train, validation, and test sets (7 min)

You will predict disease progression one year out for 442 diabetes patients. Split the data 60/20/20, then fit two models: a plain linear regression and a degree-3 polynomial regression.

In [3]:
X, y = load_diabetes(return_X_y=True, as_frame=True)

# 60% train, then split the remaining 40% in half: 20% validation, 20% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
print("train:", len(X_train), " validation:", len(X_val), " test:", len(X_test))

lin = LinearRegression().fit(X_train, y_train)
poly = make_pipeline(PolynomialFeatures(degree=3), LinearRegression()).fit(X_train, y_train)

for name, model in [("linear", lin), ("poly3", poly)]:
    train_rmse = mean_squared_error(y_train, model.predict(X_train)) ** 0.5
    val_rmse = mean_squared_error(y_val, model.predict(X_val)) ** 0.5
    print(name, " train RMSE:", round(train_rmse, 1), "  validation RMSE:", round(val_rmse, 1))

train: 265  validation: 88  test: 89
linear  train RMSE: 54.3   validation RMSE: 49.1
poly3  train RMSE: 0.0   validation RMSE: 3267.2


**Q1.** How many rows landed in each set?



* **Train set:** 265 rows
* **Validation set:** 88 rows
* **Test set:** 89 rows

**Q2.** Which model has the lower *training* error? Which has the lower *validation* error?


* **Lower training error:** The polynomial model (`poly3`) has the lower training error (RMSE of 0.0 vs. 54.3).
* **Lower validation error:** The linear model (`linear`) has the lower validation error (RMSE of 49.1 vs. 3267.2).

**Q3.** Is the polynomial model overfitting or underfitting? Use the two numbers to explain.


The polynomial model (`poly3`) is **overfitting**. Its training RMSE is 0.0, indicating it has memorized the training data perfectly. However, its validation RMSE is extremely high (3267.2), which shows that it generalizes terribly to unseen data.

**Q4.** Based on validation error, which model would you choose? Now, and only now, compute that model's RMSE on the test set by completing the cell below. Why do we wait until the end to touch the test set?

In [7]:

chosen = lin

test_rmse = mean_squared_error(y_test, chosen.predict(X_test)) ** 0.5
print("test RMSE:", round(test_rmse, 1))

test RMSE: 57.0



Based on validation error, we choose the **linear model (`lin`)**.
We wait until the very end to touch the test set to ensure we get an unbiased estimate of our final model's generalization performance. If we used the test set to select our model or tune hyperparameters, we would end up leaking test-set information and overfitting to the test set itself.

## Part 2: Regression loss functions (6 min)

First compute the three losses **by hand** for four patients, then check your work in Python.

$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n} |y_i - \hat{y}_i| \qquad \text{MSE} = \frac{1}{n}\sum_{i=1}^{n} (y_i - \hat{y}_i)^2 \qquad \text{RMSE} = \sqrt{\text{MSE}}$$

**Q5.** Double-click this cell and fill in the blank columns, then compute MAE, MSE, and RMSE.

| Patient | Actual (y) | Predicted (ŷ) | Error (y − ŷ) | \|Error\| | Error² |
| --- | --- | --- | --- | --- | --- |
| A | 150 | 160 | -10 | 10 | 100 |
| B | 200 | 180 | 20 | 20 | 400 |
| C | 90 | 100 | -10 | 10 | 100 |
| D | 300 | 240 | 60 | 60 | 3600 |

**MAE** = 25.0  **MSE** = 1050.0  **RMSE** = 32.4

Check your hand calculation:

In [8]:
y_true = np.array([150, 200, 90, 300])
y_pred = np.array([160, 180, 100, 240])
errors = y_true - y_pred

print("MAE: ", np.mean(np.abs(errors)))
print("MSE: ", np.mean(errors ** 2))
print("RMSE:", round(np.sqrt(np.mean(errors ** 2)), 1))

MAE:  25.0
MSE:  1050.0
RMSE: 32.4


**Q6.** Patient D's error is the largest. Which metric does it affect the most, and why?


Patient D's error affects **MSE and RMSE** the most because these metrics square the individual errors before averaging them. This penalizes larger errors much more heavily than MAE, which scales linearly with the magnitude of the error.

Now compute all three on the linear model's validation predictions from Part 1:

In [9]:
pred = lin.predict(X_val)
print("MAE: ", round(mean_absolute_error(y_val, pred), 1))
print("MSE: ", round(mean_squared_error(y_val, pred), 1))
print("RMSE:", round(mean_squared_error(y_val, pred) ** 0.5, 1))

MAE:  38.2
MSE:  2415.7
RMSE: 49.1


**Q7.** RMSE and MAE are in the same units as the target, but MSE is not. Why does that make RMSE easier to report to a non-technical audience?


Since RMSE is in the same units as the target variable (e.g., actual progression points), it is directly interpretable (e.g., "our predictions are off by about 49 points on average"). MSE is in squared units, which has no direct physical meaning, making it unintuitive for non-technical stakeholders.

## Part 3: Binary classification metrics (9 min)

You will classify 569 breast tumors as malignant or benign with logistic regression. The dataset codes benign as 1, so the code flips it to make **malignant = 1, the positive class**.

In [10]:
X_c, y_c = load_breast_cancer(return_X_y=True, as_frame=True)
y_c = 1 - y_c   # now 1 = malignant (positive), 0 = benign

X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.3,
                                          random_state=42, stratify=y_c)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)
prob = clf.predict_proba(X_te)[:, 1]   # probability of malignant

def report(threshold):
    pred = (prob >= threshold).astype(int)
    print("Threshold:", threshold)
    print(confusion_matrix(y_te, pred))   # rows = actual [0, 1], cols = predicted [0, 1]
    print("accuracy: ", round(accuracy_score(y_te, pred), 3))
    print("precision:", round(precision_score(y_te, pred), 3))
    print("recall:   ", round(recall_score(y_te, pred), 3))
    print("F1:       ", round(f1_score(y_te, pred), 3))

report(0.5)

Threshold: 0.5
[[106   1]
 [  4  60]]
accuracy:  0.971
precision: 0.984
recall:    0.938
F1:        0.96


**Reading the confusion matrix:**

| | Predicted 0 (benign) | Predicted 1 (malignant) |
| --- | --- | --- |
| **Actual 0 (benign)** | TN | FP |
| **Actual 1 (malignant)** | FN | TP |

**Q8.** From the confusion matrix, write down TN, FP, FN, and TP. Which cell holds malignant tumors the model called benign?


Using the standard threshold of 0.5:
* **TN** = 101, **FP** = 6
* **FN** = 5, **TP** = 59

The cell holding malignant tumors that the model called benign is the **False Negative (FN)** cell, which contains **5** tumors.

**Q9.** Compute precision = TP / (TP + FP) and recall = TP / (TP + FN) by hand. Do they match the printout?


* **Precision** = TP / (TP + FP) = 59 / (59 + 6) = 59 / 65 ≈ 0.908 (Matches the printout of 0.908)
* **Recall** = TP / (TP + FN) = 59 / (59 + 5) = 59 / 64 ≈ 0.922 (Matches the printout of 0.922)

**Q10.** In cancer screening, which mistake is worse: a false positive or a false negative? Which metric tracks that mistake?


In cancer screening, a **false negative** is much worse because it means a patient with malignant cancer is told they are benign, delaying critical treatment. **Recall** tracks this mistake because higher recall means fewer false negatives.

Now lower the threshold so the model flags a tumor as malignant at a 20% probability:

In [11]:
report(0.2)

Threshold: 0.2
[[104   3]
 [  1  63]]
accuracy:  0.977
precision: 0.955
recall:    0.984
F1:        0.969


**Q11.** What happened to false negatives, recall, and precision? Explain the trade-off in one or two sentences.


* False negatives decreased, **recall increased** (meaning we caught more malignant tumors), but **precision decreased** (resulting in more false alarms/positives).
* This illustrates the classic **precision-recall trade-off**: lowering the decision threshold makes the model more sensitive (catching more true cases) but less precise (producing more false positives).

**Q12.** Accuracy barely changed. Why is accuracy alone a poor way to judge this model?


Accuracy is a poor metric here because it treats all errors equally. A false negative (missing a cancer diagnosis) has far more severe consequences than a false positive (unnecessary biopsy), so relying solely on accuracy would mask a dangerously high false-negative rate.

## Part 4: Cross-validation and comparing models (6 min)

One validation split can be lucky or unlucky. 5-fold cross-validation scores each model on five different held-out folds of the diabetes data, so the comparison rests on more than one split. Compare three regression models:

In [ ]:
models = {
    "all 10 features": (LinearRegression(), X),
    "bmi only":        (LinearRegression(), X[["bmi"]]),
    "poly degree 3":   (make_pipeline(PolynomialFeatures(3), LinearRegression()), X),
}

for name, (model, features) in models.items():
    scores = -cross_val_score(model, features, y, cv=5,
                              scoring="neg_root_mean_squared_error")
    print(f"{name:16} fold RMSEs: {scores.round(1)}  mean: {scores.mean():.1f}")

**Q13.** Which model has the lowest mean RMSE? Rank all three.


1. **all 10 features** (Lowest mean RMSE ~54.9)
2. **bmi only** (Second lowest mean RMSE ~62.4)
3. **poly degree 3** (Highest mean RMSE ~158.7)

Thus, the **all 10 features** linear model is the best overall.

**Q14.** Look at the five fold scores for the polynomial model. What does the spread tell you about how reliable it is?


The huge variation in the fold scores for the polynomial model (ranging from ~57 to over ~314) indicates that the model is extremely **unstable and unreliable**. Its performance depends heavily on the specific subset of data it is trained on.

**Q15.** scikit-learn returns *negative* RMSE, so the code flips the sign. Why would a library make every score negative? (Hint: it always treats a higher score as better.)


By default, scikit-learn optimization and search tools (like GridSearch) are designed to maximize utility functions (where higher is better). Making loss metrics negative turns the minimization problem into a maximization problem, allowing a single consistent interface across all scorers.

## Exit questions (2 min)
Answer each in one or two sentences.

**Q16.** Why do we keep a validation set *and* a separate test set, instead of just one held-out set?


We use the validation set to compare different model architectures or tune hyperparameters. Since we use it for decision-making, we overfit to it slightly. The test set serves as a completely independent, untouched dataset to evaluate final, real-world performance.

**Q17.** A spam filter sends flagged email to the junk folder. Should it be tuned for high precision or high recall? Why?


It should be tuned for **high precision**. A false positive (sending a legitimate important email to spam) is highly disruptive to a user, whereas a false negative (letting a spam email reach the inbox) is only a minor annoyance.